In [30]:

import os
import time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from sklearn.metrics import accuracy_score, confusion_matrix
from google.colab import userdata

In [31]:
# CONFIGURATION

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
DATASET_PATH = "yelp.csv"
SAMPLE_SIZE = 5
MODEL_NAME = "gemini-2.5-flash"
# Initialize the new Client
client = genai.Client(api_key=GEMINI_API_KEY)

In [32]:

# 2. Define Structured Output Schema

class ReviewRating(BaseModel):
    explanation: str = Field(
        ...,
        description="A brief reasoning for the rating, citing specific sentiment from the text."
    )
    predicted_stars: int = Field(
        ...,
        description="The predicted star rating as an integer between 1 and 5."
    )

In [33]:
# 3. Data Loading & Preparation

def load_and_sample_data(path, n_samples):
    try:
        # Attempt to read CSV
        df = pd.read_csv(path)


        cols = df.columns.str.lower()
        if 'text' in cols and 'stars' in cols:

             pass
        elif 'review_text' in cols and 'rating' in cols:
            df = df.rename(columns={'review_text': 'text', 'rating': 'stars'})
        else:
            # Fallback for generic Kaggle structure if needed
            print(f"Columns found: {df.columns}. Attempting to use index 0 as stars, index 1 as text.")
            df.columns.values[0] = 'stars'
            df.columns.values[1] = 'text'

        # Sample data
        if len(df) > n_samples:
            df_sample = df.sample(n=n_samples, random_state=42).reset_index(drop=True)
        else:
            df_sample = df.reset_index(drop=True)

        print(f"Successfully loaded {len(df_sample)} reviews.")
        return df_sample

    except Exception as e:
        print(f"Warning: Could not load '{path}' ({e}). Generating MOCK DATA for testing.")
        return pd.DataFrame({
            'text': [
                "The food was absolutely amazing and service was top notch!",
                "Terrible experience. Cold food and rude staff.",
                "It was okay, nothing special but not bad either.",
                "Great atmosphere, but the waiting time was too long.",
                "Best pizza in town! Highly recommended."
            ],
            'stars': [5, 1, 3, 3, 5]
        })

df = load_and_sample_data(DATASET_PATH, SAMPLE_SIZE)

Successfully loaded 5 reviews.


In [34]:

# 4. Prompt Engineering Strategies

# Approach 1: Zero-Shot
# The schema does the heavy lifting for formatting.
# The prompt focuses purely on the classification task.
def prompt_zero_shot(review_text):
    return f"""
    Analyze the sentiment of the following Yelp review and predict the star rating (1-5).

    Review: "{review_text}"
    """

# Approach 2: Few-Shot
# We provide examples in the prompt text to guide the logic.
def prompt_few_shot(review_text):
    return f"""
    Classify the Yelp review into a 1-5 star rating. Use these examples as a guide for your logic:

    Review: "The wait was long, but the food was delicious."
    -> Rating: 4 (Food quality outweighs wait)

    Review: "Completely inedible. I want a refund."
    -> Rating: 1 (Strong negative)

    Review: "It was decent. Not great, not terrible."
    -> Rating: 3 (Neutral/Average)

    Now analyze this review:
    Review: "{review_text}"
    """

# Approach 3: Chain-of-Thought (CoT)
# We explicitly ask the model to deliberate in the 'explanation' field first.
def prompt_cot(review_text):
    return f"""
    You are an expert critic. Determine the rating by following these steps:
    1. Analyze the 'Service' mentioned in the text.
    2. Analyze the 'Food Quality'.
    3. Analyze the 'Value for Money'.
    4. Weigh these factors to determine the final score.

    IMPORTANT: Write your step-by-step analysis in the 'explanation' field first, then assign the 'predicted_stars'.

    Review: "{review_text}"
    """

In [35]:
# 5. Execution Engine


def get_prediction(text, prompt_func, method_name):
    prompt = prompt_func(text)

    try:

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=ReviewRating,
                temperature=0.1
            )
        )

        # Automatic parsing into the Pydantic object
        result: ReviewRating = response.parsed

        return {
            "Method": method_name,
            "Review Sample": text[:50] + "...",
            "Predicted": result.predicted_stars,
            "Explanation": result.explanation,
            "Valid_JSON": True,
            "Raw_Error": None
        }

    except Exception as e:
        print(f"Error in {method_name}: {e}")
        return {
            "Method": method_name,
            "Review Sample": text[:50] + "...",
            "Predicted": 0,
            "Explanation": "API/Parse Error",
            "Valid_JSON": False,
            "Raw_Error": str(e)
        }

def run_experiment_batch(dataframe, prompt_func, method_name):
    print(f"--- Running {method_name} ---")
    results = []

    for _, row in dataframe.iterrows():
        # Rate limit handling (simple sleep)
        time.sleep(0.5)

        res = get_prediction(row['text'], prompt_func, method_name)
        res['Actual'] = row['stars']
        results.append(res)

    return pd.DataFrame(results)

In [36]:

# 6. Run All Experiments

# 1. Zero-Shot
df_zero = run_experiment_batch(df, prompt_zero_shot, "Zero-Shot")

# 2. Few-Shot
df_few = run_experiment_batch(df, prompt_few_shot, "Few-Shot")

# 3. Chain-of-Thought
df_cot = run_experiment_batch(df, prompt_cot, "Chain-of-Thought")

# Combine
all_results = pd.concat([df_zero, df_few, df_cot], ignore_index=True)

--- Running Zero-Shot ---
Error in Zero-Shot: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
--- Running Few-Shot ---
Error in Few-Shot: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
Error in Few-Shot: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 40.872955312s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quo

In [29]:

# 7. Evaluation & Comparison


def calculate_metrics(method_df):

    valid_data = method_df[method_df['Valid_JSON'] == True]

    if valid_data.empty:
        return 0.0, 0.0

    # Accuracy
    acc = accuracy_score(valid_data['Actual'], valid_data['Predicted'])

    # Reliability (Valid JSON rate)
    validity = method_df['Valid_JSON'].mean()

    return round(acc * 100, 2), round(validity * 100, 2)

summary_data = []
for method in ["Zero-Shot", "Few-Shot", "Chain-of-Thought"]:
    subset = all_results[all_results['Method'] == method]
    acc, val = calculate_metrics(subset)
    summary_data.append({
        "Approach": method,
        "Accuracy (%)": acc,
        "JSON Validity (%)": val,
        "Sample Size": len(subset)
    })

summary_df = pd.DataFrame(summary_data)

print("\n\n====== FINAL COMPARISON TABLE ======")
print(summary_df)




====== FINAL COMPARISON TABLE ======
           Approach  Accuracy (%)  JSON Validity (%)  Sample Size
0         Zero-Shot         66.67               60.0            5
1          Few-Shot         50.00               40.0            5
2  Chain-of-Thought          0.00                0.0            5
